# HerBERT-large — macierz transferu 3×3 na PEŁNYCH danych (Kaggle)

**Wymaga:** GPU T4×2, Internet ON, dataset `pl-emotion-processed`.

In [ ]:
# transformers <5.2 — w 5.2 usunięto warmup_ratio z TrainingArguments.
# herbert_large_epochs padł na tym 10.08.2026). Górne ograniczenie utrzymuje
# recepturę identyczną z wcześniejszymi runami tej kampanii.
!pip install -q -U "transformers>=4.44,<5.2" "datasets>=2.20" accelerate 2>/dev/null
import torch,transformers; print(transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
import glob, warnings, numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
warnings.filterwarnings("ignore")
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RANDOM_STATE=42; torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
MODEL="allegro/herbert-large-cased"; tok=AutoTokenizer.from_pretrained(MODEL)
OUT="/kaggle/working"
def fc(n): return glob.glob(f"/kaggle/input/**/{n}",recursive=True)[0]
def load(name,col):
    out={}
    for sp in ["train","val","test"]:
        d=pd.read_csv(fc(f"{name}_{sp}.csv")); d[col]=d[col].fillna("")
        out[sp]={"text":d[col].tolist(),"y":d[EMOTIONS].values}
    return out
DATA={"TW":load("twitteremo","tekst"),"GO":load("go_emotions","text_pl"),"CE":load("clarin_emo","tekst")}
print({k:len(DATA[k]["train"]["y"]) for k in DATA})

In [ ]:
def opt_thr(yt,yp):
    thr=np.full(len(EMOTIONS),0.5)
    for i in range(len(EMOTIONS)):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            v=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if v>bf: bf,bt=v,t
        thr[i]=bt
    return thr
def make_ds(text,y):
    d=Dataset.from_dict({"text":text,"labels":y.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=128),batched=True,remove_columns=["text"])
class WT(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),pos_weight=self.pw.to(out.logits.device))
        return (loss,out) if return_outputs else loss
@torch.no_grad()
def predict(trainer,text):
    return expit(trainer.predict(make_ds(text,np.zeros((len(text),len(EMOTIONS))))).predictions)
def train_on(dom,epochs=3,batch=8):
    tr=DATA[dom]["train"]; y=tr["y"]; pos=y.sum(0); neg=len(y)-pos
    pw=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
    args=TrainingArguments(output_dir=f"{OUT}/m_{dom}",eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,per_device_train_batch_size=batch,
        per_device_eval_batch_size=32,gradient_accumulation_steps=2,gradient_checkpointing=True,num_train_epochs=epochs,
        learning_rate=2e-5,warmup_ratio=0.1,weight_decay=0.01,fp16=True,logging_steps=100,report_to="none",seed=RANDOM_STATE)
    cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
    tr_obj=WT(model=model,args=args,train_dataset=make_ds(tr["text"],y),eval_dataset=make_ds(DATA[dom]["val"]["text"],DATA[dom]["val"]["y"]),
        data_collator=DataCollatorWithPadding(tok),compute_metrics=cm,pos_weight=pw,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    tr_obj.train()
    thr=opt_thr(DATA[dom]["val"]["y"],predict(tr_obj,DATA[dom]["val"]["text"]))
    return tr_obj,thr

In [ ]:
rows=[]; M=pd.DataFrame(index=list(DATA),columns=list(DATA),dtype=float)
for trd in DATA:
    print(f"\n#### TRAIN {trd} (pełne dane) ####",flush=True)
    trainer,thr=train_on(trd)
    for evd in DATA:
        pred=(predict(trainer,DATA[evd]["test"]["text"])>=thr).astype(int)
        f1=f1_score(DATA[evd]["test"]["y"],pred,average="macro",zero_division=0)
        M.loc[trd,evd]=f1; rows.append({"train":trd,"eval":evd,"f1_macro":round(f1,3)})
        print(f"   {trd}->{evd}: {f1:.3f}")
    del trainer; torch.cuda.empty_cache()
    M.to_csv(f"{OUT}/herbert_large_3x3_full.csv"); pd.DataFrame(rows).to_csv(f"{OUT}/herbert_large_3x3_full_long.csv",index=False)
print("\nMacierz HerBERT-large (pełne dane):"); display(M.round(3))

## Wynik
`herbert_large_3x3_full.csv` (macierz) + `_long.csv`.